# INFO 2950 Project — Phase II

**Group:** Kai Harashima (kh862), Skye Moore (sbm232)

**Project:** Analyzing chess games from the Lichess and Chess.com APIs

**Due:** Oct. 15 (peer review), Oct. 20 (final grading)

---

## Setup

*Imports and configuration used throughout the notebook.*

In [13]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


---
# Part 1: Data Collection and Exploratory Data Analysis

## 1. Research Question(s)

How does the probability of blundering depend on how much time a player has? How
does that relationship change between bullet, blitz, rapid, and classical time modes?

## 2. Data Collection and Cleaning

*Initial draft of the data cleaning appendix. Document every step that takes the raw data file(s) and turns them into the analysis-ready dataset. Include text cells describing data collection (downloading, scraping, APIs, etc.) and text cells describing any additional curation/cleaning (merging data frames, filtering, transformations of variables, etc.). Include code cells for curation/cleaning, but not collection.*

*Note: Save data in intermediate files at several points so you are not starting from scratch every time something changes.*

### 2.1 Data Collection

*Describe how the raw data was obtained (sources, APIs, parameters, dates collected). Text only — no collection code here.*

- **Source(s):**
    - Bullet: https://lichess.org/tournament/qOuXu5XQ
    - Blitz: https://lichess.org/tournament/RFD0ylGd
    - Rapid: https://lichess.org/tournament/mvHPAYHc
    - Classical: https://lichess.org/tournament/jqxM88Wl

- **Collection method:** python script
- **Date(s) collected:** 10/18
- **Raw file location(s):** ../raw-data

### 2.2 Loading Raw Data

*Read in the raw data file(s).*

In [14]:
# Load raw data

total = []

games_bullet = pd.read_json("../data/raw/qOuXu5XQ.ndjson",lines=True)
games_blitz = pd.read_json("../data/raw/RFD0ylGd.ndjson",lines=True)
games_rapid = pd.read_json("../data/raw/mvHPAYHc.ndjson",lines=True)
games_classical = pd.read_json("../data/raw/jqxM88Wl.ndjson",lines=True)

total.append(games_bullet)
total.append(games_blitz)
total.append(games_rapid)
total.append(games_classical)
full_df = pd.concat(total, ignore_index=True)

print(full_df.shape)
full_df.head()

(29428, 17)


,id,rated,variant,speed,perf,createdAt,lastMoveAt,status,source,players,winner,opening,moves,clocks,analysis,arenaTour,clock
0,Xl7ZoEek,True,standard,bullet,bullet,1791831524597,1791831590607,mate,arena,"{'white': {'user': {'name': 'Jg2405', 'id': 'j...",white,"{'eco': 'C20', 'name': 'King's Pawn Game: Beye...",e4 e5 d4 d5 dxe5 d4 Nf3 Bg4 Qxd4 Qxd4 Nxd4 c5 ...,"[3003, 3003, 2955, 2979, 2835, 2875, 2707, 282...","[{'eval': 18}, {'eval': 22}, {'eval': -11}, {'...","{'id': 'qOuXu5XQ', 'name': 'Weekly Bullet Arena'}","{'initial': 60, 'increment': 0, 'totalTime': 60}"
1,oKqBjl1V,True,standard,bullet,bullet,1791831524597,1791831554618,mate,arena,"{'white': {'user': {'name': 'Plini2020', 'id':...",black,"{'eco': 'C20', 'name': 'Center Game', 'ply': 3}",e4 e5 d4 f5 dxe5 fxe4 Nc3 g6 Be3 Bg7 Qd2 Ne7 O...,"[6003, 6003, 5947, 5947, 5891, 5859, 5843, 580...",NaN,"{'id': 'qOuXu5XQ', 'name': 'Weekly Bullet Arena'}","{'initial': 60, 'increment': 0, 'totalTime': 60}"
2,XpFCVDQo,True,standard,bullet,bullet,1791831507010,1791831583075,resign,arena,"{'white': {'user': {'name': 'IcelordBg', 'flai...",black,"{'eco': 'A43', 'name': 'Benoni Defense: Old Be...",d4 c5 c4 cxd4 Qxd4 Nc6 Qd1 Nf6 Nc3 e6 a3 d5 cx...,"[6003, 6003, 6003, 5987, 5939, 5987, 5923, 588...","[{'eval': 15}, {'eval': 61}, {'eval': 0, 'best...","{'id': 'qOuXu5XQ', 'name': 'Weekly Bullet Arena'}","{'initial': 60, 'increment': 0, 'totalTime': 60}"
3,g4SUkgBB,True,standard,bullet,bullet,1791831507010,1791831545776,mate,arena,"{'white': {'user': {'name': 'LikaLia157', 'id'...",black,"{'eco': 'D00', 'name': 'Queen's Pawn Game', 'p...",d4 d5 c3 Nc6 Nf3 Bg4 e3 Nf6 Be2 e6 Nbd2 Bd6 h3...,"[6003, 6003, 6003, 5923, 6003, 5779, 6003, 564...",NaN,"{'id': 'qOuXu5XQ', 'name': 'Weekly Bullet Arena'}","{'initial': 60, 'increment': 0, 'totalTime': 60}"
4,PNjEGGkv,True,standard,bullet,bullet,1791831507010,1791831537227,mate,arena,"{'white': {'user': {'name': 'gmcg44', 'id': 'g...",white,"{'eco': 'C50', 'name': 'Four Knights Game: Ita...",e4 e5 Nc3 Nc6 Nf3 Nf6 Bc4 Bc5 d3 O-O O-O d6 Nd...,"[6003, 6003, 6003, 5971, 5979, 5923, 5931, 588...",NaN,"{'id': 'qOuXu5XQ', 'name': 'Weekly Bullet Arena'}","{'initial': 60, 'increment': 0, 'totalTime': 60}"


### 2.3 Cleaning Steps

*Describe each cleaning/curation step and why it was needed.*

**Step 1:** we cleaned the code


In [15]:
# Cleaning step 1
def get_white_rating(p):
    return p["white"]["rating"]
def get_black_rating(p):
    return p["black"]["rating"]
def either_berserk(p):
    return p["white"].get("berserk", False) or p["black"].get("berserk", False)
def either_provisional(p):
    return p["white"].get("provisional", False) or p["black"].get("provisional", False)
def get_clock_initial(c):
    return c["initial"]
def get_clock_increment(c):
    return c["increment"]

games_df = full_df.copy()
games_df["white_rating"]    = games_df["players"].apply(get_white_rating)
games_df["black_rating"]    = games_df["players"].apply(get_black_rating)
games_df["berserk"]         = games_df["players"].apply(either_berserk)
games_df["provisional"]     = games_df["players"].apply(either_provisional)
games_df["clock_initial"]   = games_df["clock"].apply(get_clock_initial)
games_df["clock_increment"] = games_df["clock"].apply(get_clock_increment)

**Step 2:** 


In [16]:
# Cleaning step 2

#drop games without analysis
games_df = games_df.loc[games_df["analysis"].notna()]

#drop berserk/provisional
games_df = games_df.loc[~games_df["berserk"] & ~games_df["provisional"]]

#drop unneeded columns:
games_df = games_df.drop(columns=["rated", "variant", "source", "perf", "arenaTour",
                                  "createdAt", "lastMoveAt", "opening",
                                  "players", "clock", "berserk", "provisional"])
games_df = games_df.rename(columns={"id": "game_id"})
games_df = games_df.reset_index(drop=True)
print(games_df.shape)
games_df.head()

(2894, 11)


,game_id,speed,status,winner,moves,clocks,analysis,white_rating,black_rating,clock_initial,clock_increment
0,XpFCVDQo,bullet,resign,black,d4 c5 c4 cxd4 Qxd4 Nc6 Qd1 Nf6 Nc3 e6 a3 d5 cx...,"[6003, 6003, 6003, 5987, 5939, 5987, 5923, 588...","[{'eval': 15}, {'eval': 61}, {'eval': 0, 'best...",2716,2444,60,0
1,euCaFbR9,bullet,outoftime,white,f4 d5 Nf3 c5 g3 Nc6 Bg2 Bg4 O-O e6 h3 Bxf3 exf...,"[6003, 6003, 6003, 5771, 6003, 5603, 6003, 514...","[{'eval': -23}, {'eval': -17}, {'eval': -24}, ...",1773,1188,60,0
2,eptF8Xmh,bullet,resign,white,d4 d6 Nf3 Nf6 g3 h5 Bg2 g6 h4 Nbd7 b3 Bg7 Bb2 ...,"[6003, 6003, 5891, 5963, 5835, 5963, 5787, 594...","[{'eval': 15}, {'eval': 60}, {'eval': 48}, {'e...",2198,2135,60,0
3,RrP8uNYV,bullet,mate,white,d4 g5 c4 g4 Nc3 Bg7 e4 c5 d5 Bxc3+ bxc3 Qa5 f3...,"[6003, 6003, 6003, 5963, 5987, 5907, 5979, 581...","[{'eval': 15}, {'eval': 179, 'best': 'g8f6', '...",1769,1586,60,0
4,IeMfp0On,bullet,mate,white,d4 Nf6 c4 e6 Nc3 d5 cxd5 exd5 Bg5 Bg4 Bxf6 Qxf...,"[6003, 6003, 6003, 5979, 5923, 5979, 5835, 597...","[{'eval': 15}, {'eval': 15}, {'eval': 16}, {'e...",1330,1801,60,0


**Step 3:** 

In [17]:
# Cleaning step 3
def get_judgment(entry):
    if "judgment" in entry:
        return entry["judgment"]["name"]
    return "None"
def game_to_df(game):
    moves = game["moves"].split()
    n = len(moves)
    start = game["clock_initial"] * 100
    clock_before = [start, start] + list(game["clocks"][:n - 2])
    df = pd.DataFrame({"ply": range(1, n + 1), "clock_before": clock_before})
    df["game_id"] = game["game_id"]
    df["speed"] = game["speed"]
    df["color"] = (df["ply"] % 2 == 1).map({True: "white", False: "black"})
    df["rating"] = df["color"].map({"white": game["white_rating"],
                                    "black": game["black_rating"]})
    df["pct_time_remaining"] = 100 * df["clock_before"] / start

    df["judgment"] = pd.Series(game["analysis"][:n]).apply(get_judgment)
    df["judgment"] = df["judgment"].replace(np.nan, "None")
    return df

move_frames = games_df.apply(game_to_df, axis=1)
moves_df = pd.concat(list(move_frames), ignore_index=True)
moves_df = moves_df.loc[:, ["game_id", "speed", "ply", "color", "rating",
                            "pct_time_remaining", "judgment"]]
moves_df["is_blunder"] = (moves_df["judgment"] == "Blunder").astype(int)
print(moves_df.shape)
print(moves_df["judgment"].value_counts())
moves_df.head()

(196955, 8)
judgment
None          156932
Inaccuracy     17679
Blunder        13987
Mistake         8357
Name: count, dtype: int64


,game_id,speed,ply,color,rating,pct_time_remaining,judgment,is_blunder
0,XpFCVDQo,bullet,1,white,2716,100.00,None,0
1,XpFCVDQo,bullet,2,black,2444,100.00,None,0
2,XpFCVDQo,bullet,3,white,2716,100.05,Inaccuracy,0
3,XpFCVDQo,bullet,4,black,2444,100.05,None,0
4,XpFCVDQo,bullet,5,white,2716,100.05,None,0


### 2.4 Saving the Analysis-Ready Dataset

*Write the cleaned dataset to `data/processed/`.*

In [18]:
# Save cleaned data
games_out = games_df.drop(columns=["moves", "clocks", "analysis"])

games_out.to_csv("../data/processed/games.csv", index=False)
moves_df.to_csv("../data/processed/moves.csv", index=False)

print("games:", games_out.shape, "| moves:", moves_df.shape)


games: (2894, 8) | moves: (196955, 8)


## 3. Data Description

*Initial draft of the data description, about the **analysis-ready** data. Inspired by the format in Gebru et al., 2018 ("Datasheets for Datasets"), sections 3.1–3.5.*

**What are the observations (rows) and the attributes (columns)?**

- We have two tables. In games.csv, there is one row per game. We have 2894 rows; each contains game id, speed (time control), status, winner, both ratings, starting clock, and ply count. In moves.csv, there one row per ply (a ply is one player's individual move, two plys are one "full" move). moves.csv has 196,955 rows containing speed, ply number, color, mover's rating, percent of starting time remaining, engine judgment, and a blunder indicator (0 or 1).

**Why was this dataset created?**

- We created this dataset to test if the probability of a blunder changes based on time remaining, and if the relationship differs across bullet, blitz, rapid, and classical time controls.

**How did you create this dataset? What tools, procedures, and data sources did you use?**

- We used Python to pull the Lichess tournament games as NDJSON files for four weekly arenas. We then turned into dataframes and cleaned using pandas (see section 2).

**What processes might have influenced what data was observed and recorded and what was not?**

- We have only included arena tournament games, which limits our selection to players who chose to enter tournaments, and also limited our scope to games where someone requested engine analysis (we couldn't tell what was a blunder otherwise). We removed games with a berserked or provisionally rated player (berserked means one player had half the normal time, which interferes with our correlation of time and blunders. Provisionally rated means that the players' elo is uncertain and so could interfere with our visualization of how the time/blunder correlation changes with elo).

**What preprocessing was done, and how did the data come to be in the form that you are using?**

- See section 2: We dropped columns that weren't needed for our analysis, flattened the player and clock nested dicts into individual columns containing the relevant info, filtered out berserk/provisional/unanalyzed games, and created a new dataset for moves, containing one row per ply and also normalizing the clock to percent time remaining to adjust for the difference in scale between different time controls.

**If people are involved, were they aware of the data collection and if so, what purpose did they expect the data to be used for?**

- Lichess users already agree to public game data under the site's terms. We dropped usernames from the data, as they weren't relevant to our study.

**Where can your raw source data be found, if applicable?** *(Provide a link to the raw data hosted on GitHub, Cornell Google Drive, or Cornell Box.)*

- 

### 3.1 Dataset Overview

*Show the structure of the analysis-ready dataset (shape, columns, dtypes, head).*

In [19]:
# Dataset overview


## 4. Data Limitations

*Identify potential problems with the dataset: harms, data gaps, biases, and how these could affect the meaning of results and the impact of results on people.*

- 

## 5. Exploratory Data Analysis

*Perform an initial EDA. Compute relevant summary statistics and show plots applicable to the research question(s). Choose analyses deliberately (no "chart barf") and explain each one. Use the EDA to confirm/update the research question(s) and the data description.*

### 5.1 Summary Statistics

*Typical values of relevant single variables.*

In [20]:
# Summary statistics


### 5.2 Single-Variable Distributions

*Explain what is plotted and why.*

In [21]:
# Univariate plots


### 5.3 Relationships Between Variables

*Connections between pairs of relevant variables. Explain what is plotted and why.*

In [22]:
# Bivariate plots


### 5.4 EDA Takeaways

*What did the EDA reveal? Did it change the research question(s) or data description?*

- 

## 6. Questions for Reviewers (Part 1)

*Specific questions for peer reviewers and the project mentor, specific to the dataset and based on group discussion.*

1. 
2. 
3. 

---
# Part 2: Preregistration of Analyses

*Preregister at least two analyses that will be presented in the final report regardless of the results. Each statement must include (1) a hypothesis and (2) the analysis that will be run to support it. Do not run these analyses first and then "preregister" them. Remember that the final report must include at least one model showing patterns or relationships between variables that addresses the research question.*

## Preregistration Statement 1

**Hypothesis 1:** 

**Analysis:** 

## Preregistration Statement 2

**Hypothesis 2:** 

**Analysis:** 

## Questions for Reviewers (Part 2)

*Specific questions for the project mentor to answer, looking toward the next stage of the project.*

1. 
2. 
3. 